[<img src="../quantumsymmetry_logo.png" alt="QuantumSymmetry" width="450"/>](https://github.com/dariopicozzi/quantumsymmetry)

> **Note:** if you are running this notebook on Google Colab, the next cell will install `quantumsymmetry` and its dependencies (this might take a few minutes):

In [1]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip -q install quantumsymmetry
    !pip -q install pylatexenc

# Spin adaptation: exact total spin on the tree

Particle number, spin projection $S_z$ and point-group symmetry are *diagonal* in the computational basis, so `MinimalCircuit` imposes them exactly through the choice of support. **Total spin $S^2$ is the exception:** its eigenstates, the *configuration state functions* (CSFs), are entangled combinations of determinants, so a determinant support generally mixes spins and the plain tree reaches a definite-spin state only variationally.

The `total_spin` keyword of `MinimalCircuit.from_particle_number` removes that last approximation: it prepares a state of exact total spin $S$ at **every** parameter value, on the unchanged determinant-tree circuit and with the diagonal metric intact. We use the lithium hydride (4, 4) active space from the [VQE tutorial](tree_03_vqe.ipynb).

## The determinant support is spin-mixed

The $\{N = 4,\, S_z = 0\}$ sector of the (4, 4) active space has 36 determinants. As a support they fix particle number and $S_z$ exactly, but not total spin: the sector splits into 20 singlets ($S = 0$), 15 triplets ($S = 1$) and one quintet ($S = 2$). A plain tree over these leaves is therefore a spin *mixture* -- at generic angles its $\langle S^2\rangle$ is nowhere near zero.

In [2]:
import numpy as np
from quantumsymmetry import MinimalCircuit, total_spin_expectation

m = 4                                       # spatial orbitals in the (4, 4) CAS
det = MinimalCircuit.from_particle_number(m, (2, 2))      # plain determinant tree

rng = np.random.default_rng(0)
s2 = [total_spin_expectation(det.statevector(rng.uniform(-2, 2, det.num_parameters)), m)
      for _ in range(200)]
print('determinants in support :', len(det.support))
print(f'<S^2> at random angles   : min {min(s2):.2f}, mean {np.mean(s2):.2f}, max {max(s2):.2f}')

determinants in support : 36
<S^2> at random angles   : min 0.09, mean 0.96, max 2.15


## A spin eigenstate at every parameter

Passing `total_spin=0` builds the tree over the 20 singlet CSFs instead. Its state is $\mathbf c = U_{\mathrm S}\,\mathbf a(\boldsymbol\theta)$, where $U_{\mathrm S}$ is the fixed determinant-to-CSF change of basis, applied *classically* and never as a gate. Because $U_{\mathrm S}$ is fixed, the Fubini-Study metric stays **exactly diagonal**; because every leaf is a singlet, $\langle S^2\rangle = 0$ at every parameter value. The circuit is the same determinant tree -- the same CNOTs -- with only the singlet directions left free.

In [3]:
mc = MinimalCircuit.from_particle_number(m, (2, 2), total_spin=0)

theta = rng.uniform(-2, 2, mc.num_parameters)
Ginv = mc.inverse_metric(theta)
print('free parameters   :', mc.num_parameters, '(singlets)   vs',
      det.num_parameters, '(determinants)')
print('CNOTs             :', mc.circuit.count_ops().get('cx', 0))
print('metric off-diag   :', f'{np.abs(Ginv - np.diag(np.diag(Ginv))).max():.1e}')
print('<S^2> at random th:', f'{mc.total_spin_expectation(theta):.1e}')

free parameters   : 19 (singlets)   vs 35 (determinants)
CNOTs             : 84
metric off-diag   : 0.0e+00
<S^2> at random th: 1.5e-35


## Spin-adapted VQE on LiH

We minimise the molecular energy by natural-gradient descent on the CSF tree -- the same loop as [tutorial 3](tree_03_vqe.ipynb), now confined to the singlet subspace. The Hamiltonian is the (4, 4) active space on the $2m$ blocked Jordan-Wigner qubits (the basis the spin-adapted tree lives in). LiH has a closed-shell singlet ground state, so we benchmark against the **PySCF CASCI(4, 4)** energy -- an independent reference for the exact answer in this active space. The optimiser reaches it to within chemical accuracy while $\langle S^2\rangle$ stays pinned at machine zero.

In [4]:
from quantumsymmetry import minimize_energy
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper
from pyscf import gto, scf, mcscf

atom = 'Li 0 0 0; H 0 0 1.5957'

# (4, 4) active-space Hamiltonian on the 2m blocked Jordan-Wigner qubits. The
# mapped operator is the active *electronic* energy, so we keep the constant
# shift (nuclear repulsion + inactive core) to report total energies.
problem = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=m).transform(
    PySCFDriver(atom=atom, basis='sto-3g').run())
H = np.real(JordanWignerMapper().map(problem.hamiltonian.second_q_op()).to_matrix())
shift = sum(problem.hamiltonian.constants.values())

# independent reference: PySCF CASCI(4, 4) total energy
casci = mcscf.CASCI(scf.RHF(gto.M(atom=atom, basis='sto-3g', verbose=0)).run(), m, 4)
casci.verbose = 0
e_casci = casci.kernel()[0]

# H is a dense matrix, so we pass an explicit energy callback; the Hamiltonian
# slot is left None (it feeds only the built-in estimator path, unused here).
def energy_fn(theta):
    psi = mc.statevector(theta)
    return float(psi @ (H @ psi))

result = minimize_energy(mc, None, energy_fn=energy_fn)
energy = result.energy + shift

print('VQE energy    :', round(energy, 8), 'Ha')
print('CASCI (PySCF) :', round(e_casci, 8), 'Ha')
print('error         :', f'{abs(energy - e_casci):.2e}', 'Ha')
print('<S^2>         :', f'{mc.total_spin_expectation(result.optimal_parameters):.1e}')
print('parameters    :', mc.num_parameters, '  CNOTs:', mc.circuit.count_ops().get('cx', 0))

VQE energy    : -7.86314135 Ha
CASCI (PySCF) : -7.86319778 Ha
error         : 5.64e-05 Ha
<S^2>         : 6.6e-66
parameters    : 19   CNOTs: 84


The spin-adapted ansatz reaches the singlet ground state within chemical accuracy with $\langle S^2\rangle$ at machine zero, on **19 of the determinant tree's 35 parameters** (the non-singlet directions are gone) and the same CNOT count. Spin is a property of the basis, not of the parameters, so no angle can leave the sector; and because the change of basis is a *fixed* unitary, the diagonal metric and the shift-rule gradients carry over unchanged. A different `total_spin` targets any other spin sector in exactly the same way.

### The series

1. <a href="tree_01_welcome.ipynb" />Welcome: the binary-tree ansatz</a>
2. <a href="tree_02_ansatz_and_metric.ipynb" />The ansatz, its pruning and its diagonal metric</a>
3. <a href="tree_03_vqe.ipynb" />Natural-gradient VQE for molecules</a>
4. <a href="tree_04_dynamics.ipynb" />Real-time evolution</a>
5. <a href="tree_05_sampling.ipynb" />Sector-Haar sampling</a>
6. <a href="tree_06_dressing.ipynb" />Dressing the ansatz: a symmetry-adapted Schrieffer-Wolff transformation</a>
7. <a href="tree_07_spin.ipynb" />Spin adaptation: exact total spin on the tree</a>

## The same singlet, on fewer qubits: the symmetry-adapted encoding

Everything above ran in the plain $2m$-qubit Jordan–Wigner space. The spin-adapted tree also composes with the **symmetry-adapted encoding**: handing an `Encoding` (which tapers particle number, spin projection and the point group) to `from_encoding(..., total_spin=0)` lands the *same* spin-pure CSF tree natively on the tapered qubits. Each intermediate Slater determinant is mapped to its encoded correspondent, so $\langle S^2\rangle = 0$ is preserved exactly while the circuit now acts on `enc.encoded_qubits` qubits instead of $2m$.

In [5]:
from quantumsymmetry import Encoding

# The (4, 4) CAS in the symmetry-adapted encoding: particle number, spin
# projection and point group are tapered out, so the circuit acts on fewer
# qubits.  from_encoding(..., total_spin=0) lands the spin-pure CSF tree
# natively in that encoding.
enc = Encoding(atom, 'sto-3g', CAS=(4, m), output_format='qiskit')
mc_enc = MinimalCircuit.from_encoding(enc, total_spin=0)

H_enc = np.real(enc.hamiltonian.to_matrix())
exact = float(np.linalg.eigvalsh(H_enc)[0])          # exact encoded-sector ground

def energy_fn(theta):
    psi = mc_enc.statevector(theta)
    return float(psi @ (H_enc @ psi))

result = minimize_energy(mc_enc, None, energy_fn=energy_fn,
                         reference_basis_index=enc.hartree_fock_leaf)

print('encoded qubits :', enc.encoded_qubits, ' (vs', det.num_qubits, 'in Jordan-Wigner)')
print('parameters     :', mc_enc.num_parameters, '  CNOTs:',
      mc_enc.circuit.count_ops().get('cx', 0))
print('VQE energy     :', round(result.energy, 8), 'Ha')
print('sector ground  :', round(exact, 8), 'Ha')
print('error          :', f'{abs(result.energy - exact):.2e}', 'Ha')
print('<S^2>          :', f'{mc_enc.total_spin_expectation(result.optimal_parameters):.1e}')

encoded qubits : 5  (vs 8 in Jordan-Wigner)
parameters     : 11   CNOTs: 28
VQE energy     : -7.86319765 Ha
sector ground  : -7.86319778 Ha
error          : 1.28e-07 Ha
<S^2>          : 0.0e+00


<p style="text-align: left"> <a href="tree_06_dressing.ipynb" />&lt; Previous: Dressing the ansatz: a symmetry-adapted Schrieffer-Wolff transformation</a> </p>